# Telegram Expense Tracker — Demo evaluada

Notebook principal de la tarea final. Se ejecuta de principio a fin y en orden, sin Telegram.
Cada sección corresponde a un criterio de la rúbrica o a un bono declarado (ver `docs/bonos.md`).

| Sección | Contenido | Estado |
|---|---|---|
| 0 | Setup y ficha de reproducción | **Disponible** (modelo exacto: Etapa 3) |
| 1 | Caso de uso y criterio de éxito | **Disponible** |
| 2 | LLM y prompts | Pendiente |
| 3 | ReAct | Pendiente |
| 4 | Historial | Pendiente |
| 5 | Seguridad | Pendiente |
| 6 | Router (bono) | Pendiente |
| 7 | Memoria avanzada (bono) | Pendiente |
| 8 | Herramienta de acción (bono) | Pendiente |
| 9 | Juez LLM (bono) | Pendiente |
| 10 | Golden set (bono) | Pendiente |

## Sección 0 — Setup y ficha de reproducción

Esta sección prepara el entorno y no requiere `.env` ni red: solo muestra si cada variable está **definida** o **falta** (nunca su valor) y demuestra el trazador con un secreto falso que se enmascara. El modelo exacto se completa en la Etapa 3.

In [ ]:
import sys
from pathlib import Path

# El notebook vive en notebooks/: se agrega la raíz del repositorio al path.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Python:", sys.version.split()[0])
print("Raíz del repositorio:", ROOT.name)

In [ ]:
from app.config import config_status

# Solo presencia de variables, nunca valores.
for name, state in config_status().items():
    print(f"{name:<34} {state}")

### Demostración del trazador

Se registra un evento con un secreto **falso** construido en tiempo de ejecución. La consola y el archivo JSONL solo muestran `***`.

In [ ]:
import tempfile

from app.trace import Tracer

fake_key = "AIza" + "x" * 35  # secreto de ejemplo, no real
with tempfile.TemporaryDirectory() as tmp:
    tracer = Tracer(session="demo", trace_dir=tmp)
    tracer.record("USER_INPUT", {"texto": "Registra este recibo"})
    tracer.record("TOOL_CALL", {"tool": "analizar_recibo", "api_key": fake_key,
                                "nota": f"clave usada: {fake_key}"})
    contenido = tracer.path.read_text(encoding="utf-8")

print("Eventos:", len(tracer.events))
print("El secreto aparece en el JSONL:", fake_key in contenido)

## Sección 1 — Caso de uso y criterio de éxito

**Usuario:** persona que registra sus gastos personales y usa Telegram a diario.
**Entrada:** imagen de un recibo (JPG/PNG) y texto opcional.
**Salida:** archivo en Drive, fila nueva en Sheets y confirmación con el número de fila.

**Criterio observable:** dada una imagen válida de `data/receipts/`, el caso es exitoso cuando
(1) fecha, comercio y monto coinciden con `data/receipts/expected.json`;
(2) la categoría es una de las permitidas;
(3) Drive devuelve un `file_id` y un `web_view_link` que existen;
(4) la planilla tiene exactamente una fila más, con ese enlace;
(5) la respuesta contiene el número de fila.

**Por qué un LLM:** los recibos no tienen estructura fija y el flujo exige decidir entre registrar, pedir confirmación, pedir otra foto o rechazar.
**Por qué herramientas:** el LLM no puede persistir; Drive y Sheets devuelven observaciones verificables.

La celda siguiente muestra el documento completo del caso de uso.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
display(Markdown((ROOT / "docs" / "use_case.md").read_text(encoding="utf-8")))